<a href="https://colab.research.google.com/github/WingmanShi/WingmanShi-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*



Now, i gonona look at each table in detail.

Prompt: "print each tables name, the number of rows and the first three records respectively."

I'm expecting to see each table's name, its total row count, and the first 3 rows of data displayed to understand the structure of the raw datasets.

In [19]:
from IPython.display import display

for name, df in tables.items():
    print(f"Table: {name} | Rows: {df.shape[0]:,}")
    print("First 3 records:")
    display(df.head(3))
    print("-" * 40)

Table: trips | Rows: 570
First 3 records:


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,Casual
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual


----------------------------------------
Table: bikes | Rows: 60
First 3 records:


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,Electric,2024


----------------------------------------
Table: stations | Rows: 11
First 3 records:


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0


----------------------------------------


That seems make sense.

Now, i gonna check for missing values in each table.

Prompt: "now, check for missing values for each table."

I'm expecting to see a breakdown of the number of null or missing values across every column in each of the three tables.

In [17]:
for name, df in tables.items():
    print(f"Table: {name}")
    print(df.isnull().sum())
    print("-" * 30)

Table: trips
trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time          91
duration_min         0
user_type            0
dtype: int64
------------------------------
Table: bikes
bike_id          0
bike_type        0
purchase_year    0
dtype: int64
------------------------------
Table: stations
station_id      0
station_name    0
zone            0
capacity        0
dtype: int64
------------------------------


There are 30 missing values in duration_min and 1 in capacity.

Now, i will check the primary key's uniqueness.

Prompt: "show me for each table the number of rows and the number of unique ids. the unique ids should be trip_id for trips; bike_id for bikes and station_id for stations"

I'm expecting to see the primary key ID counts compared against the total row counts to see if there are any duplicate records or missing keys.

In [20]:
for name, df in tables.items():
    if name == "trips":
        unique_col = "trip_id"
    elif name == "bikes":
        unique_col = "bike_id"
    elif name == "stations":
        unique_col = "station_id"

    print(f"Table: {name}")
    print(f"  Rows: {df.shape[0]:,}")
    print(f"  Unique {unique_col}: {df[unique_col].nunique():,}")
    print("-" * 30)

Table: trips
  Rows: 570
  Unique trip_id: 570
------------------------------
Table: bikes
  Rows: 60
  Unique bike_id: 60
------------------------------
Table: stations
  Rows: 11
  Unique station_id: 11
------------------------------


trip_id is noe unique in the trips table, so now i gonna look at this table.

Prompt: "fint items in trips that share trip_id. i.e. if trip_id = t0001 is  in the table twice, print both these rows."

I'm expecting to see any rows that share the same trip_id displayed side by side so I can inspect the duplication.

In [21]:
trips = tables["trips"]

# Filter for rows where the trip_id appears more than once
duplicates = trips[trips.duplicated(subset=["trip_id"], keep=False)]

# Sort by trip_id to easily inspect them side by side
duplicates = duplicates.sort_values(by="trip_id")

from IPython.display import display
display(duplicates)

,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

I have found some missing values in the above part, now let's drop them.

Prompt: "drop the missing values."

I'm expecting to see the updated row counts for the trips and stations tables after successfully removing rows with missing values.

In [8]:
# Drop rows where 'duration_min' or station 'capacity' is missing
cleaned_trips = tables["trips"].dropna(subset=["duration_min"]).copy()
cleaned_stations = tables["stations"].dropna(subset=["capacity"]).copy()

# Update our tables dictionary with the cleaned versions
tables["trips"] = cleaned_trips
tables["stations"] = cleaned_stations

print(f"Trips rows after dropping missing: {len(tables['trips'])}")
print(f"Stations rows after dropping missing: {len(tables['stations'])}")

Trips rows after dropping missing: 582
Stations rows after dropping missing: 11


Now, let's deal with the duplicate data.

Prompt: "please drop duplicate rows based on 'trip_id', keeping the first occurrence."

I'm expecting to see the trips row count decrease after dropping duplicate trip IDs while retaining the first recorded instance of each.

In [9]:
# Drop duplicate rows based on 'trip_id', keeping the first occurrence
cleaned_trips_no_dupes = tables["trips"].drop_duplicates(subset=["trip_id"], keep="first").copy()

# Update our tables dictionary
tables["trips"] = cleaned_trips_no_dupes

print(f"Trips rows after dropping duplicates: {len(tables['trips'])}")

Trips rows after dropping duplicates: 570


What's more, there're inconsistent casing in user_type and bike_type, let's standardize them.

Prompt: "please standardize bike_type and user_type casing to Title Case."

I'm expecting to see the unique values for bike types and user types cleaned up into consistent Title Case formatting.

In [10]:
# Standardize bike_type casing to Title Case (e.g., 'Electric', 'Standard')
tables["bikes"]["bike_type"] = tables["bikes"]["bike_type"].str.title()

# Standardize user_type casing to Title Case (e.g., 'Casual', 'Member')
tables["trips"]["user_type"] = tables["trips"]["user_type"].str.title()

print("Bike types unique values:", tables["bikes"]["bike_type"].unique())
print("User types unique values:", tables["trips"]["user_type"].unique())

Bike types unique values: ['Standard' 'Electric ' 'Electric' 'E-Bike']
User types unique values: ['Casual' 'Member' 'Casual ']


Also, the date format is inconsistent, so let's fix it.

Prompt: "please standardize the data format."

I'm expecting to see the start_time column successfully parsed into uniform datetime objects and a count of any unparseable values converted to NaT.

In [11]:
# Strip any leading/trailing whitespaces in string columns to keep categories clean
tables["bikes"]["bike_type"] = tables["bikes"]["bike_type"].str.strip()
tables["trips"]["user_type"] = tables["trips"]["user_type"].str.strip()

# Standardize 'start_time' to a uniform datetime format
# errors='coerce' will safely turn any unparsable strings into NaT (Not a Time) so they don't break your code
tables["trips"]["start_time"] = pd.to_datetime(tables["trips"]["start_time"], errors="coerce")

# Check if there are any parsing errors (NaT values)
print("Null or unparsed start_times:", tables["trips"]["start_time"].isnull().sum())
print("Sample start_times:")
print(tables["trips"]["start_time"].head())

Null or unparsed start_times: 91
Sample start_times:
0   2025-03-01 05:52:00
1   2025-03-01 06:31:00
2   2025-03-01 07:06:00
3   2025-03-01 07:06:00
4   2025-03-01 07:36:00
Name: start_time, dtype: datetime64[ns]


I find another problem, 'Electric ' has a trailing space and 'E-Bike' is a different label for what should probably just be 'Electric'. If we don't fix this, 'Electric' and 'E-Bike' will be treated as separate groups.

Prompt: "please strip any stray spaces in bike_type and map 'E-Bike' to 'Electric'."

I'm expecting to see the unique bike types clean up to just standard categories like 'Standard' and 'Electric' with no trailing spaces or alternate labels.

In [22]:
# Strip any stray spaces in bike_type
tables["bikes"]["bike_type"] = tables["bikes"]["bike_type"].str.strip()

# Map 'E-Bike' to 'Electric' so categories are consistent
tables["bikes"]["bike_type"] = tables["bikes"]["bike_type"].replace({"E-Bike": "Electric"})

print("Cleaned Bike Types:", tables["bikes"]["bike_type"].unique())

Cleaned Bike Types: ['Standard' 'Electric']


now, i want to see the tables after cleaning for chekcing if i clean them properly.

Prompt: "print the tables' first 5 records to see the table after we fix."

I'm expecting to see the first 5 rows of all three tables printed out to visually verify that the cleaning steps were applied correctly.

In [23]:
from IPython.display import display

for name, df in tables.items():
    print(f"Cleaned Table: {name} (Rows: {df.shape[0]:,})")
    display(df.head(5))
    print("=" * 40)

Cleaned Table: trips (Rows: 570)


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,Casual
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


Cleaned Table: bikes (Rows: 60)


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,Electric,2024
3,B004,Standard,2021
4,B005,Standard,2022


Cleaned Table: stations (Rows: 11)


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0


ok, now everything makes sense. let's join and aggregate them!

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

Before writing the SQL join, we need to make sure DuckDB is pointing to our freshly cleaned tables (trips, bikes, stations) instead of the old raw ones. Re-registering them ensures that any missing values, duplicates, or weird bike types we cleaned out are fully reflected in our SQL queries.

Prompt: "please re-register each cleaned table into DuckDB."

In [24]:
# Re-register each cleaned table into DuckDB so our SQL uses the cleaned data
for name, df in tables.items():
    con.register(name, df)

print("DuckDB successfully updated with cleaned tables!")

DuckDB successfully updated with cleaned tables!


ok, it's cleaned tables now, we can join them now. The business brief asks for the average trip duration by bike type and by the zone of the start station. So I gonna join trips to bikes using bike_id to look up the bike_type; and join trips to stations using start_station_id = station_id to look up the starting station's zone. Finally, group by those two attributes (bike_type and zone) and calculate the total number of trips (COUNT) and the average duration (AVG).

Prompt: "please join trips to bikes using bike_id to look up the bike_type; and join trips to stations using start_station_id = station_id to look up the starting station's zone. Last, group by those two attributes (bike_type and zone) and calculate the total number of trips (COUNT) and the average duration (AVG)."

In [25]:
from IPython.display import display

# SQL query to join the tables and aggregate by bike type and start zone
query = """
SELECT
    b.bike_type,
    s.zone AS start_zone,
    COUNT(t.trip_id) AS number_of_trips,
    ROUND(AVG(t.duration_min), 2) AS avg_duration_min
FROM trips AS t
JOIN bikes AS b ON t.bike_id = b.bike_id
JOIN stations AS s ON t.start_station_id = s.station_id
GROUP BY b.bike_type, s.zone
ORDER BY b.bike_type, s.zone;
"""

final_result = q(query)
display(final_result)

,bike_type,start_zone,number_of_trips,avg_duration_min
0,Electric,Centre,66,14.42
1,Electric,East,11,15.73
2,Electric,North,23,15.96
3,Electric,South,24,13.42
4,Electric,West,22,15.68
5,Standard,Centre,185,34.50
6,Standard,East,20,19.50
7,Standard,North,39,20.36
8,Standard,South,60,43.15
9,Standard,West,73,19.82


## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*

- I could not verify whether the 91 records with unparsed start_time values (which were coerced to NaT) showed any systematic time-of-day or day-of-week patterns that might bias our duration averages.

- I could not verify whether the duplicate trip_id rows represented true duplicate system logs, multi-leg rides, or data entry errors, meaning my choice to keep the first occurrence may have omitted unique context.

- I could not verify if the missing values in duration_min were missing at random or linked to specific stations or bike types, which could slightly skew the final aggregated averages.